In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [2]:
#A. EXTRACK
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("Jumlah baris df_transaksi:", df_transaksi.count(), "baris")
df_transaksi.printSchema()

print("Jumlah baris df_produk:", df_produk.count(), "baris")
df_produk.printSchema()

print("Jumlah baris df_ulasan:", df_ulasan.count(), "baris")
df_ulasan.printSchema()

Jumlah baris df_transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Jumlah baris df_produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Jumlah baris df_ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [4]:
# TRANSFORM — Penggabungan
df_gabungan = df_transaksi.join(df_ulasan, on="order_id", how="left")
df_gabungan = df_gabungan.join(df_produk, on="product_id", how="inner")

# Menambahkan kolom total_pendapatan
df_gabungan = df_gabungan.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

df_gabungan.show()

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

In [5]:
#C. TRANSFORM — Penanganan Data Kosong & Pengayaan 
df_gabungan = df_gabungan.withColumn("ada_ulasan", col("rating").isNotNull())

df_gabungan = df_gabungan.na.fill({"rating": 0})

df_gabungan.show()

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|     false|
|        10|     TX5|   

mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan"?
jawab: Nilai rating yang kosong diisi dengan angka 0 karena transaksi tersebut belum memiliki ulasan. Angka 0 digunakan sebagai penanda bahwa belum ada rating karena rating dimulai dari 1-5 sementara 0 bukan diantara angka tersebut. Angka aatau rating 0 ini juga bukan sebagai rating buruk dari pelanggan.

In [6]:
#LOAD
!hdfs dfs -mkdir -p /user/irkham/tugas6

df_gabungan.write.mode("overwrite").partitionBy("kategori").parquet(
    f"hdfs://localhost:9000/user/irkham/tugas6/hasil_etl"
)

print("Data berhasil disimpan ke HDFS.")
!hdfs dfs -ls -R /user/irkham/tugas6/hasil_etl

df_simpan = spark.read.parquet("hdfs://localhost:9000/user/irkham/tugas6/hasil_etl")
print("Jumlah baris hasil akhir dari HDFS:", df_simpan.count())

Data berhasil disimpan ke HDFS.
-rw-r--r--   3 irkham supergroup          0 2026-09-25 20:43 /user/irkham/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - irkham supergroup          0 2026-09-25 20:43 /user/irkham/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 irkham supergroup      16366 2026-09-25 20:43 /user/irkham/tugas6/hasil_etl/kategori=Elektronik/part-00000-c3c6872f-1dc7-4ae9-bcd4-97de19a3bb42.c000.snappy.parquet
drwxr-xr-x   - irkham supergroup          0 2026-09-25 20:43 /user/irkham/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 irkham supergroup      11133 2026-09-25 20:43 /user/irkham/tugas6/hasil_etl/kategori=Fashion/part-00000-c3c6872f-1dc7-4ae9-bcd4-97de19a3bb42.c000.snappy.parquet
drwxr-xr-x   - irkham supergroup          0 2026-09-25 20:43 /user/irkham/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 irkham supergroup      10392 2026-09-25 20:43 /user/irkham/tugas6/hasil_etl/kategori=Kesehatan/part-00000-c3c6872f-1dc7-4ae9-bcd4-97de19a3bb42.c000.snappy.parquet
drwxr-

In [10]:
# Insight Akhir
df_insight = df_gabungan.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    spark_sum(when(col("ada_ulasan") == True, 1).otherwise(0)).alias("jumlah_ada_ulasan")
).withColumn(
    "persentase_ada_ulasan", col("jumlah_ada_ulasan") / col("total_transaksi") * 100
).orderBy("persentase_ada_ulasan")

df_insight.show()

+------------+---------------+-----------------+---------------------+
|    kategori|total_transaksi|jumlah_ada_ulasan|persentase_ada_ulasan|
+------------+---------------+-----------------+---------------------+
|     Makanan|            536|              369|    68.84328358208955|
|   Kesehatan|            961|              662|    68.88657648283039|
|     Fashion|           1035|              726|    70.14492753623188|
|Rumah Tangga|            815|              575|     70.5521472392638|
|  Elektronik|           1653|             1168|      70.659407138536|
+------------+---------------+-----------------+---------------------+



mengapa hal ini mungkin penting diketahui oleh tim marketing?
jawab: Rendahnya persentase ulasan pada kategori Makanan menunjukkan minimnya umpan balik dari pelanggan. Hal ini penting diketahui oleh tim marketing karena feedback pelanggan dapat membantu membangun kepercayaan, meningkatkan kepuasan, dan mengevaluasi kualitas produk. Berdasarkan informasi tersebut, tim marketing dapat membuat kampanye atau program yang menarik minat pelanggan untuk memberikan ulasan.
